In [32]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel



from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
import optuna



In [2]:
df = pd.read_csv("santander-customer-transaction-prediction/train.csv")
df

,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,train_2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,train_3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,train_4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,train_199995,0,11.4880,-0.4956,8.2622,3.5142,10.3404,11.6081,5.6709,15.1516,...,6.1415,13.2305,3.9901,0.9388,18.0249,-1.7939,2.1661,8.5326,16.6660,-17.8661
199996,train_199996,0,4.9149,-2.4484,16.7052,6.6345,8.3096,-10.5628,5.8802,21.5940,...,4.9611,4.6549,0.6998,1.8341,22.2717,1.7337,-2.1651,6.7419,15.9054,0.3388
199997,train_199997,0,11.2232,-5.0518,10.5127,5.6456,9.3410,-5.4086,4.5555,21.5571,...,4.0651,5.4414,3.1032,4.8793,23.5311,-1.5736,1.2832,8.7155,13.8329,4.1995
199998,train_199998,0,9.7148,-8.6098,13.6104,5.7930,12.5173,0.5339,6.0479,17.0152,...,2.6840,8.6587,2.7337,11.1178,20.4158,-0.0786,6.7980,10.0342,15.5289,-13.9001


In [3]:
df.isnull().sum().sum()

np.int64(0)

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.dtypes.value_counts()

float64    200
object       1
int64        1
Name: count, dtype: int64

In [6]:
target_col = "target"
y = df[target_col]
X = df.drop(columns=[target_col, "ID_code"])

In [7]:
y.describe()

count    200000.000000
mean          0.100490
std           0.300653
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: target, dtype: float64

In [8]:
corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

to_drop = [
    column
    for column in upper.columns
    if any(upper[column] > 0.90)
]

print("Original features:", X.shape[1])
print("Removed features:", len(to_drop))

Original features: 200
Removed features: 0


In [9]:


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(X_train.shape)
print(X_valid.shape)
print(y_valid.shape)



(160000, 200)
(40000, 200)
(40000,)


In [20]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

X_valid_scaled

array([[-0.62858622,  2.08121814, -0.53874339, ..., -1.06325184,
         0.57574643, -0.39108186],
       [-1.03211093, -1.68992093,  1.79280274, ...,  0.39091615,
        -1.2159055 ,  0.44398807],
       [-1.59486111, -1.3049001 , -0.96688246, ..., -0.22879349,
         0.92467816,  1.4210405 ],
       ...,
       [-2.0287834 ,  1.36440801, -1.22291196, ..., -2.36205814,
        -0.0133621 ,  0.10635541],
       [-0.60670916,  1.22069523, -0.88704327, ...,  0.63378413,
        -0.30899496, -0.60425693],
       [-0.48469113,  0.19529043,  0.7796423 , ..., -1.8464658 ,
        -0.49116029, -0.02963525]], shape=(40000, 200))

In [14]:


lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [22]:
y_pred_lr = lr.predict(X_valid_scaled)


print("===== logreg =====")
print("Accuracy:", accuracy_score(y_valid, y_pred_lr))
print("precision:", precision_score(y_valid, y_pred_lr))
print("recall :", recall_score(y_valid, y_pred_lr ,zero_division=0))
print("f1:", f1_score(y_valid, y_pred_lr, zero_division=0))



===== logreg =====
Accuracy: 0.7834
precision: 0.2865416436845008
recall : 0.7753731343283582
f1: 0.4184454289166331


In [25]:
rf_selector = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)

rf_selector.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
y_pred_rf_selector = rf_selector.predict(X_valid)

print("===== rf_selector =====")
print("Accuracy:", accuracy_score(y_valid, y_pred_rf_selector))
print("precision:", precision_score(y_valid, y_pred_rf_selector))
print("recall :", recall_score(y_valid, y_pred_rf_selector ,zero_division=0))
print("f1:", f1_score(y_valid, y_pred_rf_selector, zero_division=0))



===== rf_selector =====
Accuracy: 0.8995
precision: 0.0
recall : 0.0
f1: 0.0


c:\Users\zand\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [27]:
importance = pd.Series(
    rf_selector.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("\nFeature Importance:")
print(importance)


Feature Importance:
var_81     0.017159
var_139    0.014937
var_12     0.011434
var_110    0.010708
var_53     0.010532
             ...   
var_136    0.003731
var_42     0.003725
var_185    0.003703
var_16     0.003703
var_183    0.003694
Length: 200, dtype: float64


In [33]:
selector = SelectFromModel(
    rf_selector,
    threshold="median",
    prefit=True
)

selected_mask = selector.get_support()

selected_features = X_train.columns[selected_mask]


In [34]:
X_train_selected = X_train[selected_features]
X_valid_selected = X_valid[selected_features]


print("\n========== Feature Selection ==========")
print("Before:", X_train.shape[1])
print("After :", X_train_selected.shape[1])

print("\nSelected features:")
print(list(selected_features))


========== Feature Selection ==========
Before: 200
After : 100

Selected features:
['var_0', 'var_1', 'var_2', 'var_5', 'var_6', 'var_9', 'var_12', 'var_13', 'var_18', 'var_21', 'var_22', 'var_26', 'var_31', 'var_32', 'var_33', 'var_34', 'var_35', 'var_36', 'var_40', 'var_43', 'var_44', 'var_45', 'var_48', 'var_49', 'var_51', 'var_52', 'var_53', 'var_56', 'var_67', 'var_70', 'var_71', 'var_75', 'var_76', 'var_78', 'var_80', 'var_81', 'var_82', 'var_86', 'var_87', 'var_89', 'var_90', 'var_91', 'var_92', 'var_93', 'var_94', 'var_95', 'var_99', 'var_106', 'var_107', 'var_108', 'var_109', 'var_110', 'var_112', 'var_115', 'var_118', 'var_119', 'var_121', 'var_122', 'var_123', 'var_127', 'var_128', 'var_130', 'var_131', 'var_133', 'var_137', 'var_139', 'var_141', 'var_145', 'var_146', 'var_147', 'var_148', 'var_149', 'var_150', 'var_154', 'var_155', 'var_157', 'var_162', 'var_163', 'var_164', 'var_165', 'var_166', 'var_167', 'var_169', 'var_170', 'var_172', 'var_173', 'var_174', 'var_177',

In [35]:
print(importance.head())

var_81     0.017159
var_139    0.014937
var_12     0.011434
var_110    0.010708
var_53     0.010532
dtype: float64


In [36]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_selected, y_train)

y_pred_rf = rf.predict(X_valid_selected)



In [39]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Negative:", negative)
print("Positive:", positive)
print("scale_pos_weight:", scale_pos_weight)

Negative: 143922
Positive: 16078
scale_pos_weight: 8.95148650329643


In [40]:
xgb_weight = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_weight.fit(
    X_train_selected,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [53]:
y_pred_xgb_wheight = xgb_weight.predict(X_valid_selected)

In [41]:
smote = SMOTE(random_state=42)

X_train_smote , y_train_smote = smote.fit_resample(
    X_train_selected,
    y_train
)

print("Before:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before:
target
0    143922
1     16078
Name: count, dtype: int64

After SMOTE:
target
0    143922
1    143922
Name: count, dtype: int64


In [43]:
xgb_smote = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_smote.fit(
    X_train_selected,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [54]:
y_pred_xgb_smote = xgb_smote.predict(X_valid_selected)

In [58]:
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "CatBoost"
    ],
    "Accuracy": [
        accuracy_score(y_valid, y_pred_rf),
        accuracy_score(y_valid, y_pred_xgb_wheight),
        accuracy_score(y_valid, y_pred_xgb_smote)
        
    ],
    "precision": [
         precision_score(y_valid, y_pred_rf),
         precision_score(y_valid, y_pred_xgb_wheight),
         precision_score(y_valid, y_pred_xgb_smote)
    ],
        

    "recal": [
        recall_score(y_valid, y_pred_rf ,zero_division=0),
        recall_score(y_valid, y_pred_xgb_wheight ,zero_division=0),
        recall_score(y_valid, y_pred_xgb_smote ,zero_division=0),
    ],

    "F1": [
        f1_score(y_valid, y_pred_rf, zero_division=0),
        f1_score(y_valid, y_pred_xgb_wheight, zero_division=0),
        f1_score(y_valid, y_pred_xgb_smote, zero_division=0)
    ]
})





results = results.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

print(results)

           Model  Accuracy  precision     recal        F1
0        XGBoost  0.855850   0.376381  0.661194  0.479697
1       CatBoost  0.911475   0.835905  0.148259  0.251849
2  Random Forest  0.899500   0.000000  0.000000  0.000000


c:\Users\zand\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
